# Module 2: Elliptic Curves

## 2.1 The Curve Equation

An elliptic curve over a field $\mathbb{F}$ is defined by the **Weierstrass equation**:

$$y^2 = x^3 + ax + b$$

with the constraint that $4a^3 + 27b^2 \neq 0$ (non-singular — no cusps or self-intersections).

### Bitcoin's curve: secp256k1

$$y^2 = x^3 + 7 \pmod{P}$$

where $a = 0$, $b = 7$ (a **Koblitz curve** — the zero $a$ enables optimizations).

In [12]:
# secp256k1 parameters — the numbers that secure Bitcoin

# Field prime: coordinates live in F_P
SECP_P = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F

# Group order: scalars (private keys) live in Z_N  
SECP_N = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141

# Generator point G: the "starting point" for all key generation
SECP_GX = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
SECP_GY = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8

# Curve coefficients
A_COEFF = 0
B_COEFF = 7

print("=== secp256k1 Parameters ===")
print(f"Curve: y² = x³ + {A_COEFF}x + {B_COEFF}")
print(f"P = 2²⁵⁶ - 2³² - 977")
print(f"  = {SECP_P}")
print(f"  ({SECP_P.bit_length()} bits)")
print(f"\nN = {SECP_N}")
print(f"  ({SECP_N.bit_length()} bits)")
print(f"\nNon-singular check: 4(0)³ + 27(7)² = {4*0**3 + 27*7**2} ≠ 0  ✓")
print(f"\nP mod 4 = {SECP_P % 4}  (enables efficient square roots)")

=== secp256k1 Parameters ===
Curve: y² = x³ + 0x + 7
P = 2²⁵⁶ - 2³² - 977
  = 115792089237316195423570985008687907853269984665640564039457584007908834671663
  (256 bits)

N = 115792089237316195423570985008687907852837564279074904382605163141518161494337
  (256 bits)

Non-singular check: 4(0)³ + 27(7)² = 1323 ≠ 0  ✓

P mod 4 = 3  (enables efficient square roots)


## 2.2 Elliptic Curves Over Finite Fields

Over the real numbers, an elliptic curve is a smooth curve. Over a **finite field** $\mathbb{F}_p$,
it becomes a discrete set of points — there are exactly $N$ of them (plus the point at infinity).

Let's visualize a small elliptic curve to build intuition before working with secp256k1's
enormous numbers.

In [13]:
# Small curve: y² = x³ + x + 1 over F_23
# (Using a=1, b=1, p=23 for a visible example)

p_small = 23
a_small, b_small = 1, 1

def is_quadratic_residue(n, p):
    """Is n a perfect square mod p? (Euler's criterion)"""
    if n % p == 0:
        return True
    return pow(n, (p - 1) // 2, p) == 1

def mod_sqrt_small(a, p):
    """Square root mod p for p ≡ 3 (mod 4)."""
    return pow(a, (p + 1) // 4, p)

points = []
for x in range(p_small):
    rhs = (x**3 + a_small * x + b_small) % p_small
    if is_quadratic_residue(rhs, p_small):
        y = mod_sqrt_small(rhs, p_small)
        points.append((x, y))
        if y != 0 and y != p_small - y:  # Two y values unless y=0
            points.append((x, p_small - y))
        elif y == 0:
            pass  # Only one point when y=0

points.sort()
print(f"=== Curve: y² = x³ + {a_small}x + {b_small} over F_{p_small} ===")
print(f"Number of points: {len(points)} (+ point at infinity = {len(points) + 1})")
print(f"\nAll points:")
for i, (x, y) in enumerate(points):
    end = '\n' if (i + 1) % 4 == 0 else '   '
    print(f"  ({x:2d}, {y:2d})", end=end)
print()

# ASCII visualization
print(f"\n{'─' * 50}")
print(f"Visual (x across, y up):")
grid = [['·' for _ in range(p_small)] for _ in range(p_small)]
for x, y in points:
    grid[p_small - 1 - y][x] = '■'

for row_idx, row in enumerate(grid):
    y_val = p_small - 1 - row_idx
    if y_val % 4 == 0:
        print(f"{y_val:2d} |{''.join(row)}")
print(f"   +{'─' * p_small}")
print(f"    ", end='')
for x in range(p_small):
    if x % 4 == 0:
        print(f"{x}", end=' ' * (4 - len(str(x))))
print()

=== Curve: y² = x³ + 1x + 1 over F_23 ===
Number of points: 27 (+ point at infinity = 28)

All points:
  ( 0,  1)     ( 0, 22)     ( 1,  7)     ( 1, 16)
  ( 3, 10)     ( 3, 13)     ( 4,  0)     ( 5,  4)
  ( 5, 19)     ( 6,  4)     ( 6, 19)     ( 7, 11)
  ( 7, 12)     ( 9,  7)     ( 9, 16)     (11,  3)
  (11, 20)     (12,  4)     (12, 19)     (13,  7)
  (13, 16)     (17,  3)     (17, 20)     (18,  3)
  (18, 20)     (19,  5)     (19, 18)   

──────────────────────────────────────────────────
Visual (x across, y up):
20 |···········■·····■■····
16 |·■·······■···■·········
12 |·······■···············
 8 |·······················
 4 |·····■■·····■··········
 0 |····■··················
   +───────────────────────
    0   4   8   12  16  20  


### Observation

Notice the **vertical symmetry** — for every point $(x, y)$ there's a point $(x, p-y)$.
This is because if $y^2 \equiv c \pmod{p}$, then $(-y)^2 \equiv c \pmod{p}$ too.
The negation of a point is its vertical mirror: $-P = (x, -y)$.

---